In [2]:
import torch
import torch.nn as nn
import numpy as np

# dataset
testo = "ciao come stai sto bene grazie"
parole=testo.split()
#associamo le parole a dei numeri
vocabolario = {parola: i for i, parola in enumerate(set(parole))}
#trasformiamo i numeri in parole
vocabolario_inverso = {i: parola for parola,i in vocabolario.items()}

#preparazione dati
coppie_train = []
for i in range(len(parole)-1):
    input_parola = parole[i]
    target_parola = parole[i+1]
    coppie_train.append((vocabolario[input_parola], vocabolario[target_parola]))

print("Parole:", parole)
print("Dataset:", coppie_train)
print("Vocabolario:", vocabolario)#<- L'ordine delle parole dipende dagli Hash values poichè set() usa hash tables




Parole: ['ciao', 'come', 'stai', 'sto', 'bene', 'grazie']
Dataset: [(5, 2), (2, 4), (4, 1), (1, 0), (0, 3)]
Vocabolario: {'bene': 0, 'sto': 1, 'come': 2, 'grazie': 3, 'stai': 4, 'ciao': 5}


In [3]:
class GPT_Micro(nn.Module):
    def __init__(self, vocab_size, embedding_dim=4):
        super().__init__()
        #embedding èl'azione di racchiudere la parola in informazioni numeriche
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.fc = nn.Linear(embedding_dim, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        x = self.fc(x)
        return x

#istanziamo il GPT_micro
vocab_size = len(vocabolario)
model = GPT_Micro(vocab_size)


print("Modello creato: ",model)
print("parametri: ",sum(p.numel() for p in model.parameters()))



Modello creato:  GPT_Micro(
  (embedding): Embedding(6, 4)
  (fc): Linear(in_features=4, out_features=6, bias=True)
)
parametri:  54


In [4]:
#conversione in tensori
inputs = torch.tensor([coppia[0] for coppia in coppie_train])
targets = torch.tensor([coppia[1] for coppia in coppie_train])

#addestramento semplificato
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.1)

print('Addestramento in corso...')
for epoch in range(100):
    optimizer.zero_grad()
    outputs = model(inputs)
    loss = criterion(outputs, targets)
    loss.backward()
    optimizer.step()

    if epoch % 20 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")


Addestramento in corso...
Epoch 0, Loss: 1.6011
Epoch 20, Loss: 0.0005
Epoch 40, Loss: 0.0000
Epoch 60, Loss: 0.0000
Epoch 80, Loss: 0.0000


In [5]:
from prompt_toolkit import prompt
#Predizione
model.eval()
with torch.no_grad():
    for i, (input_idx,target_idx) in enumerate(coppie_train):
        input_parola = vocabolario_inverso[input_idx]
        target_parola = vocabolario_inverso[target_idx]

        output = model(torch.tensor([input_idx]))
        pred_idx = output.argmax().item()
        pred_parola = vocabolario_inverso[pred_idx]

        print(f"Input: '{input_parola}' -> Predizione '{pred_parola}' (Target: '{target_parola}')")
        print(f"  {'✓' if pred_parola == target_parola else '✗'}")
        

Input: 'ciao' -> Predizione 'come' (Target: 'come')
  ✓
Input: 'come' -> Predizione 'stai' (Target: 'stai')
  ✓
Input: 'stai' -> Predizione 'sto' (Target: 'sto')
  ✓
Input: 'sto' -> Predizione 'bene' (Target: 'bene')
  ✓
Input: 'bene' -> Predizione 'grazie' (Target: 'grazie')
  ✓


In [6]:
def genera_frase(frase_input, max_parole=10):

    parole = frase_input.split()
    
    if len(parole) == 0:
        return "Inserisci almeno una parola!"
    
    # Controlla che tutte le parole siano nel vocabolario
    for parola in parole:
        if parola not in vocabolario:
            return f"Parola '{parola}' non nel vocabolario!"
    
    frase_generata = parole.copy()
    
    with torch.no_grad():
        for _ in range(max_parole - len(parole)):
            # Prendi l'ultima parola
            ultima_parola = frase_generata[-1]
            input_idx = vocabolario[ultima_parola]
            input_tensor = torch.tensor([input_idx])
            
            # Predici prossima parola
            output = model(input_tensor)
            pred_idx = output.argmax().item()
            prossima_parola = vocabolario_inverso[pred_idx]
            
            # Aggiungi alla frase
            frase_generata.append(prossima_parola)
            
            # Se predice una parola già vista, fermati
            if prossima_parola in frase_generata[:-1]:
                break
    
    return " ".join(frase_generata)

# Test con frasi
print("\n=== GENERAZIONE FRASI ===")
frasi_test = [
    "ciao",
    "ciao come",
    "sto bene"
]

for frase in frasi_test:
    risultato = genera_frase(frase, max_parole=6)
    print(f"Input: '{frase}'")
    print(f"Output: '{risultato}'")
    print("-" * 40)



=== GENERAZIONE FRASI ===
Input: 'ciao'
Output: 'ciao come stai sto bene grazie'
----------------------------------------
Input: 'ciao come'
Output: 'ciao come stai sto bene grazie'
----------------------------------------
Input: 'sto bene'
Output: 'sto bene grazie stai sto'
----------------------------------------


In [7]:
# Dopo l'addestramento (punto 5), aggiungi:

def predici_prossima_parola(parola_input):
    """Predice la prossima parola dato un input"""
    if parola_input not in vocabolario:
        return f"Parola '{parola_input}' non nel vocabolario!"
    
    # Converti in tensore
    input_idx = vocabolario[parola_input]
    input_tensor = torch.tensor([input_idx])
    
    # Fai la predizione
    with torch.no_grad():
        output = model(input_tensor)
        pred_idx = output.argmax().item()
    
    return vocabolario_inverso[pred_idx]

# Test interattivo
print("\n=== TEST INTERATTIVO ===")
print("Vocabolario disponibile:", list(vocabolario.keys()))

# Prova con parole del vocabolario
parole_test = ["ciao", "come", "stai", "sto", "bene"]
for parola in parole_test:
    predizione = predici_prossima_parola(parola)
    print(f"Input: '{parola}' → Predizione: '{predizione}'")



=== TEST INTERATTIVO ===
Vocabolario disponibile: ['bene', 'sto', 'come', 'grazie', 'stai', 'ciao']
Input: 'ciao' → Predizione: 'come'
Input: 'come' → Predizione: 'stai'
Input: 'stai' → Predizione: 'sto'
Input: 'sto' → Predizione: 'bene'
Input: 'bene' → Predizione: 'grazie'


In [ ]:
#interfaccia interattiva
print("\nGPT Micro interattivo")
print("Exit = termina il programma")
print("Parole conosciute:", sorted(vocabolario.keys()))

while True:
    user = input("\nInserisci una parola conosciuta").strip()

    if user.lower() == "exit":
        print("Arrivederci")
        break

    if not user:
        print("inserisci parola o frase")
        continue

    parole_input = user.split()

    #singola parola
    if len(parole_input) == 1:
        parola = parole_input[0]
        if parola in vocabolario:
            predizione = predici_prossima_parola(parola)
            print(f"\nInput: {parola} -> Predizione: {predizione}")

            #mostra le probabilità
            with torch.no_grad():
                input_idx = vocabolario[parola]
                input_tensor = torch.tensor([input_idx])
                output = model(input_tensor)
                probabilità = torch.softmax(output, dim = 1)[0]

                print("\nProbabilità per ogni parola:")
                for idx, prob in enumerate(probabilità):
                    parola_voc = vocabolario_inverso[idx]
                    print(parola_voc , " : ", round(prob.item()*100, 1), "%")
        
        else:
            print(user, ": parola non presente nel vocabolario")
    
    else:
        risultato = genera_frase(user, max_parole=10)
        print(f"input {user}")
        print(f"Frase Generata: {risultato}")

# APPRENDIMENTO:
# Durante i 100 epoch di addestramento, per ogni coppia (ciao → come):
# Il modello fa una predizione (inizialmente sbagliata)
# La CrossEntropyLoss misura quanto è sbagliata
# Il backpropagation corregge tutti i numeri della tabella embedding e della matrice lineare
# Ripeti → i numeri convergono verso valori che fanno predire le risposte giuste
# In sostanza, i pesi del modello (54 parametri totali) sono stati "regolati" finché per ogni parola in input 
# la parola successiva corretta ottiene il punteggio più alto.


GPT Micro interattivo
Exit = termina il programma
Parole conosciute: ['bene', 'ciao', 'come', 'grazie', 'stai', 'sto']
bena : parola non presente nel vocabolario
Input: bene -> Predizione: grazie

Probabilità per ogni parola:
bene  :  0.0 %
sto  :  0.0 %
come  :  0.0 %
grazie  :  100.0 %
stai  :  0.0 %
ciao  :  0.0 %
Input: grazie -> Predizione: stai

Probabilità per ogni parola:
bene  :  4.7 %
sto  :  0.0 %
come  :  0.0 %
grazie  :  36.0 %
stai  :  58.3 %
ciao  :  1.0 %
Input: ciao -> Predizione: come

Probabilità per ogni parola:
bene  :  0.0 %
sto  :  0.0 %
come  :  100.0 %
grazie  :  0.0 %
stai  :  0.0 %
ciao  :  0.0 %
Input: sto -> Predizione: bene

Probabilità per ogni parola:
bene  :  100.0 %
sto  :  0.0 %
come  :  0.0 %
grazie  :  0.0 %
stai  :  0.0 %
ciao  :  0.0 %
Input: stai -> Predizione: sto

Probabilità per ogni parola:
bene  :  0.0 %
sto  :  100.0 %
come  :  0.0 %
grazie  :  0.0 %
stai  :  0.0 %
ciao  :  0.0 %
Arrivederci


In [16]:
#VERSIONE CON CONTESTO IN MEMORIA
class MicroGPTchat:
    def __init__(self,model,vocab,vocab_inv):
        self.model = model
        self.vocab = vocab
        self.vocab_inv = vocab_inv
        self.context=[]

    def reset_context(self):
        #resetta memoria
        self.context=[]
        print("l'ho dimenticato 🐧")

    def chat(self,input_text):
        #chat contestuale
        parole=input_text.split()
        self.context.extend(parole)
        #predici basandoti sull'ultima parola in contesto
        if not self.context:
            return "inserisci qualcosa!"
        
        ultima_parola = self.context[-1]
        if ultima_parola not in self.vocab:
            return ultima_parola," parola sconosciuta"

        with torch.no_grad():
            input_idx = self.vocab[ultima_parola]
            input_tensor = torch.tensor([input_idx])
            output = self.model(input_tensor)
            pred_idx = output.argmax().item()
            prossima_parola = self.vocab_inv[pred_idx]

        #aggiungi predizione al contesto
        self.context.append(prossima_parola)

        return prossima_parola

print("\n=== CHAT CON MEMORIA ===")
bot = MicroGPTchat(model, vocabolario, vocabolario_inverso)

# Test
messaggi = ["ciao", "come", "stai"]
for msg in messaggi:
    risposta = bot.chat(msg)
    print(f"Tu: {msg}")
    print(f"Bot: {risposta}")
    print(f"Contesto: {bot.context}")
    print()


=== CHAT CON MEMORIA ===
Tu: ciao
Bot: come
Contesto: ['ciao', 'come']

Tu: come
Bot: stai
Contesto: ['ciao', 'come', 'come', 'stai']

Tu: stai
Bot: sto
Contesto: ['ciao', 'come', 'come', 'stai', 'stai', 'sto']

